# Plan-and-Execute [Step 08.03 - Commit to a plan, then adapt]

> **MLCourse - Agentic AI - LangGraph**

ReAct decides the next step after seeing the last observation. Plan-and-Execute
inverts that:

```
  PLANNER   -> writes the WHOLE step list up front
  EXECUTOR  -> runs step 1, then step 2, ... (can be a cheaper model)
  REPLANNER -> after each step: are we done? does the rest of the plan still hold?
```

The plan is an **artifact**. You can print it, log it, show it to a user for
approval, cache it, or diff two of them. ReAct has nothing comparable.

### What you'll learn

- Separating the planner from the executor, including using different models.
- Why the **replanner** is what makes this work - not the planner.
- Managing `past_steps` so the executor has context without a full transcript.
- Failure modes: over-planning, brittle plans, replanner loops, lost objectives.

### Key takeaways

- Planning up front buys reviewability, cheaper execution, and progress tracking.
- Without a replanner the pattern is brittle - the first surprise derails it.
- The plan should be a list of **goals**, not a list of tool calls. Tool-level plans
  belong in ReWOO (notebook 04).

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                  # environment variable access
import time                                # timing + backoff sleeps
from pathlib import Path                   # locating the track root
from dotenv import load_dotenv             # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we find the track root `03_agentic_ai`,
# then load the (gitignored) .env that lives there. Every provider-touching
# notebook in this track uses exactly this block.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

GROQ_KEY = os.getenv("GROQ_API_KEY")       # never print this value
GROQ_MODEL = "qwen/qwen3.8-27b"            # fast hosted model, generous free tier
OLLAMA_MODEL = "llama3.1:8b"               # local fallback if Groq is unavailable


def make_llm(temperature: float = 0.0, max_tokens: int = 512):
    """Return a chat model. Groq first (fast, hosted); local Ollama as fallback.

    OpenAI is never used anywhere in this course.
    """
    if GROQ_KEY:
        from langchain_groq import ChatGroq
        return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                        temperature=temperature, max_tokens=max_tokens)
    from langchain_ollama import ChatOllama
    return ChatOllama(model=OLLAMA_MODEL, temperature=temperature)


def safe_invoke(model, messages, retries: int = 4, pause: float = 1.5):
    """Invoke a chat model with exponential backoff on rate limits (HTTP 429).

    Groq's free tier allows roughly 8000 tokens per minute. Teaching notebooks
    fire many small calls in a row, so a retry loop is not optional here.
    """
    delay = pause
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(pause)              # pace the next call politely
            return out
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print("  [backoff] %s -- retrying in %.1fs" % (type(exc).__name__, delay))
            time.sleep(delay)
            delay *= 2                     # exponential backoff
    raise RuntimeError("unreachable")


print("Track root :", TRACK.name)
print("Provider   :", "Groq / " + GROQ_MODEL if GROQ_KEY else "Ollama / " + OLLAMA_MODEL)


### The shared task world


In [ ]:
# Every notebook in this module attacks THE SAME task with a different reasoning
# pattern, so the comparison in notebook 05 is apples-to-apples.

from langchain_core.tools import tool

# A tiny deterministic "database". Deterministic matters: we need to check
# correctness automatically, without a human reading the answer.
POPULATION = {"tokyo": 13_960_000, "lagos": 15_400_000, "lima": 9_750_000}
AREA_KM2 = {"tokyo": 2194, "lagos": 1171, "lima": 2672}

TOOL_CALLS = {"count": 0}          # instrumentation: how many tool calls happened


@tool
def population(city: str) -> str:
    """Return the population of a city as a plain number string.

    Args:
        city: City name, e.g. "Tokyo".
    """
    TOOL_CALLS["count"] += 1
    return str(POPULATION.get(city.strip().lower(), "unknown city"))


@tool
def area_km2(city: str) -> str:
    """Return the land area of a city in square kilometres as a plain number string.

    Args:
        city: City name, e.g. "Tokyo".
    """
    TOOL_CALLS["count"] += 1
    return str(AREA_KM2.get(city.strip().lower(), "unknown city"))


TOOLS = [population, area_km2]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

TASK = (
    "Among Tokyo, Lagos and Lima, which city has the highest population density "
    "(people per square kilometre)? Answer with the city name and the density "
    "rounded to the nearest whole number."
)

# Ground truth, computed here so the notebook can grade itself.
DENSITIES = {c: POPULATION[c] / AREA_KM2[c] for c in POPULATION}
GT_CITY = max(DENSITIES, key=DENSITIES.get)
GT_DENSITY = round(DENSITIES[GT_CITY])

print("Task:", TASK)
print()
for c in sorted(DENSITIES, key=DENSITIES.get, reverse=True):
    print("  %-6s %9d / %5d = %7.0f people/km2" % (c, POPULATION[c], AREA_KM2[c], DENSITIES[c]))
print()
print("Ground truth -> %s, %d" % (GT_CITY.title(), GT_DENSITY))


def grade(answer: str) -> bool:
    """Automatic grader: the answer must name the right city AND the right density.

    The density is accepted within +/-2 to tolerate rounding differences.
    """
    import re
    if not answer:
        return False
    low = answer.lower()
    if GT_CITY not in low:
        return False
    cleaned = low.replace(",", "").replace(".", " ")
    numbers = [int(n) for n in re.findall(r"\d+", cleaned)]
    return any(abs(n - GT_DENSITY) <= 2 for n in numbers)


### Instrumentation: a token and latency meter


In [ ]:
import time


class Meter:
    """Accumulates token usage, call counts and wall-clock time for one run.

    Every pattern in this module is wrapped in one of these, so notebook 05 can
    compare them on identical instrumentation.
    """

    def __init__(self, name):
        self.name = name
        self.input_tokens = 0
        self.output_tokens = 0
        self.llm_calls = 0
        self.tool_calls = 0
        self.seconds = 0.0
        self._t0 = None

    def start(self):
        TOOL_CALLS["count"] = 0
        self._t0 = time.time()
        return self

    def stop(self):
        self.seconds = time.time() - self._t0
        self.tool_calls = TOOL_CALLS["count"]
        return self

    def record(self, message):
        """Add one AIMessage's usage to the totals, then return the message."""
        usage = getattr(message, "usage_metadata", None) or {}
        if usage:
            self.input_tokens += usage.get("input_tokens", 0)
            self.output_tokens += usage.get("output_tokens", 0)
            self.llm_calls += 1
        return message

    def record_all(self, messages):
        """Add usage from every AIMessage in a list (for create_agent results)."""
        for m in messages:
            if getattr(m, "usage_metadata", None):
                self.record(m)
        return messages

    @property
    def total_tokens(self):
        return self.input_tokens + self.output_tokens

    def report(self, answer=None, correct=None):
        print()
        print("=" * 62)
        print("PATTERN : %s" % self.name)
        print("-" * 62)
        print("LLM calls    : %d" % self.llm_calls)
        print("tool calls   : %d" % self.tool_calls)
        print("input tokens : %d" % self.input_tokens)
        print("output tokens: %d" % self.output_tokens)
        print("TOTAL tokens : %d" % self.total_tokens)
        print("latency      : %.1fs" % self.seconds)
        if correct is not None:
            print("correct      : %s" % ("YES" if correct else "NO"))
        print("=" * 62)
        if answer:
            print(answer)
        return self


### 1. The state

`plan` is the work remaining; `past_steps` is what has been done and what it
produced. Keeping them separate is what lets the executor see relevant history
without carrying a full ReAct transcript.

In [4]:
from typing import Annotated, TypedDict
import operator, re
from langgraph.graph import StateGraph, START, END
from langchain.agents import create_agent


class PlanExecState(TypedDict):
    task: str
    plan: list                                    # steps still to do
    past_steps: Annotated[list, operator.add]     # (step, result) pairs, accumulating
    response: str                                 # set when finished
    cycles: int                                   # safety counter


MAX_CYCLES = 6        # hard cap, so a confused replanner cannot spin forever

### 2. The planner

One call, up front, producing the entire step list. Two prompt rules matter:

1. **"Each step must be executable on its own"** - otherwise you get steps like
   "continue the analysis" that mean nothing in isolation.
2. **"The final step must state the answer"** - otherwise the plan completes and
   nobody has actually answered the question.

In [5]:
planner_llm = make_llm(max_tokens=320)


def plan_step(state: PlanExecState) -> dict:
    """PLANNER: produce the complete step list before any work happens."""
    prompt = (
        "You are a planner. Break the objective below into a minimal numbered list "
        "of ordered steps.\n"
        "Rules:\n"
        "- Each step must be executable on its own by a worker with these tools: "
        "population(city), area_km2(city).\n"
        "- Do not add steps that are not needed.\n"
        "- The final step must state the answer.\n"
        "- Output ONLY the numbered steps, nothing else.\n\n"
        "Objective: " + state["task"]
    )
    raw = meter.record(safe_invoke(planner_llm, prompt)).content
    steps = [re.sub(r"^\s*\d+[\.\)]?\s*", "", ln).strip()
             for ln in raw.splitlines() if ln.strip()]
    steps = [s for s in steps if len(s) > 5][:8]

    print("[planner  ] %d steps" % len(steps))
    for i, s in enumerate(steps, 1):
        print("            %d. %s" % (i, s[:95]))
    return {"plan": steps, "cycles": 0}

### 3. The executor

Runs **one** step at a time, given the objective, recent history, and the current
step. Because it only follows instructions and calls tools, it can be a smaller and
cheaper model than the planner - one of the concrete wins of splitting the roles.
(Here both use the same model so the comparison in notebook 05 stays fair.)

In [6]:
executor_agent = create_agent(
    model=make_llm(max_tokens=350),
    tools=TOOLS,
    system_prompt=("You execute ONE step of a plan. Use tools for facts. "
                   "Be brief: state only the result of this step."),
)


def execute_step(state: PlanExecState) -> dict:
    """EXECUTOR: do exactly the next step, nothing more."""
    step = state["plan"][0]

    history = ""
    if state["past_steps"]:
        history = "\nAlready completed:\n" + "\n".join(
            "- %s -> %s" % (s, r) for s, r in state["past_steps"][-4:])

    prompt = ("Overall objective: %s%s\n\nExecute ONLY this step: %s"
              % (state["task"], history, step))

    print("[executor ] %s" % step[:85])
    out = executor_agent.invoke({"messages": [("user", prompt)]})
    meter.record_all(out["messages"])
    result = out["messages"][-1].content.strip().replace("\n", " ")
    print("[executor ] -> %s" % result[:85])

    return {"past_steps": [(step, result)], "plan": state["plan"][1:]}

> **Note `"plan": state["plan"][1:]`.** The executor consumes the step it just ran.
> Forgetting this is the single most common Plan-and-Execute bug: the graph runs
> step 1 forever and only the cycle cap saves you.

### 4. The replanner - the part that actually matters

A plan written before any information was gathered is a guess. The replanner runs
after every step and answers two questions:

1. **Are we done?** If the accumulated results already answer the objective, stop and
   produce the answer. (Rigid scripts happily keep going past this point.)
2. **Does the remaining plan still make sense?** If a step turned up something
   unexpected, rewrite what is left.

Without the replanner, Plan-and-Execute is a script that breaks on the first
surprise. With it, you get planning *and* adaptivity.

In [7]:
replanner_llm = make_llm(max_tokens=320)


def replan_step(state: PlanExecState) -> dict:
    """REPLANNER: finish, or revise the remaining steps."""
    done = "\n".join("- %s -> %s" % (s, r) for s, r in state["past_steps"])
    remaining = "\n".join("- " + s for s in state["plan"]) or "(none left)"

    prompt = (
        "Objective: %s\n\nCompleted so far:\n%s\n\nRemaining planned steps:\n%s\n\n"
        "If the completed work already answers the objective, reply with exactly:\n"
        "ANSWER: <the final answer>\n"
        "Otherwise reply with the revised remaining steps as a numbered list and "
        "nothing else." % (state["task"], done, remaining)
    )
    raw = meter.record(safe_invoke(replanner_llm, prompt)).content.strip()
    cycles = state.get("cycles", 0) + 1

    if raw.upper().startswith("ANSWER:"):
        answer = raw.split(":", 1)[1].strip()
        print("[replanner] FINISH -> %s" % answer.replace("\n", " ")[:85])
        return {"response": answer, "plan": [], "cycles": cycles}

    steps = [re.sub(r"^\s*\d+[\.\)]?\s*", "", ln).strip()
             for ln in raw.splitlines() if ln.strip()]
    steps = [s for s in steps if len(s) > 5][:6]
    print("[replanner] continue with %d step(s)" % len(steps))
    return {"plan": steps, "cycles": cycles}

In [8]:
def route_after_replan(state: PlanExecState) -> str:
    if state["response"]:
        return "done"
    if not state["plan"]:
        print("[route    ] plan empty with no answer -> forcing finish")
        return "force_answer"
    if state["cycles"] >= MAX_CYCLES:
        print("[route    ] cycle budget exhausted -> forcing finish")
        return "force_answer"
    return "execute"


def force_answer(state: PlanExecState) -> dict:
    """Safety net: synthesise an answer from whatever evidence we gathered."""
    done = "\n".join("- %s -> %s" % (s, r) for s, r in state["past_steps"])
    prompt = ("Objective: %s\n\nEvidence gathered:\n%s\n\n"
              "State the final answer in one sentence." % (state["task"], done))
    return {"response": meter.record(safe_invoke(replanner_llm, prompt)).content.strip()}


pg = StateGraph(PlanExecState)
pg.add_node("planner", plan_step)
pg.add_node("executor", execute_step)
pg.add_node("replanner", replan_step)
pg.add_node("force_answer", force_answer)
pg.add_edge(START, "planner")
pg.add_edge("planner", "executor")
pg.add_edge("executor", "replanner")
pg.add_conditional_edges("replanner", route_after_replan,
                         {"execute": "executor", "force_answer": "force_answer", "done": END})
pg.add_edge("force_answer", END)

plan_exec_app = pg.compile()
print(plan_exec_app.get_graph().draw_ascii())

              +-----------+        
              | __start__ |        
              +-----------+        
                     *             
                     *             
                     *             
               +---------+         
               | planner |         
               +---------+         
                     *             
                     *             
                     *             
               +----------+        
               | executor |        
               +----------+        
                     *             
                     *             
                     *             
              +-----------+        
              | replanner |        
              +-----------+        
              ..           ..      
            ..               ..    
          ..                   ..  
+--------------+                 ..
| force_answer |               ..  
+--------------+             ..    
              **           .

### 5. Run it


In [9]:
meter = Meter("Plan-and-Execute").start()
out = plan_exec_app.invoke({
    "task": TASK, "plan": [], "past_steps": [], "response": "", "cycles": 0,
})
meter.stop()

pe_answer = out["response"]
meter.report(pe_answer, grade(pe_answer))

[planner  ] 4 steps
            1. Retrieve the population and area in square kilometers for Tokyo, Lagos, and Lima using the `pop
            2. Calculate the population density (population divided by area) for each of the three cities.
            3. Identify the city with the highest calculated population density and round its density to the n
            4. State the answer as the city name and the rounded density.
[executor ] Retrieve the population and area in square kilometers for Tokyo, Lagos, and Lima usin


[executor ] -> - Tokyo: population 13,960,000; area 2,194 km² - Lagos: population 15,400,000; area 1


[replanner] continue with 3 step(s)
[executor ] Calculate the population density (population divided by area) for each of the three c


[executor ] -> Densities (population ÷ area): - Tokyo: 13,960,000 ÷ 2,194 ≈ 6,363 people/km² - Lagos


[replanner] FINISH -> Lagos, 13151

PATTERN : Plan-and-Execute
--------------------------------------------------------------
LLM calls    : 6
tool calls   : 6
input tokens : 2566
output tokens: 539
TOTAL tokens : 3105
latency      : 16.1s
correct      : YES
Lagos, 13151


In [10]:
print("EXECUTION TRACE")
for i, (step, result) in enumerate(out["past_steps"], 1):
    print("%d. %s" % (i, step[:80]))
    print("   -> %s" % result[:100])
print()
print("replan cycles:", out["cycles"])
print("steps left   :", out["plan"] or "(none)")

EXECUTION TRACE
1. Retrieve the population and area in square kilometers for Tokyo, Lagos, and Lima
   -> - Tokyo: population 13,960,000; area 2,194 km² - Lagos: population 15,400,000; area 1,171 km² - Lima
2. Calculate the population density (population divided by area) for each of the th
   -> Densities (population ÷ area): - Tokyo: 13,960,000 ÷ 2,194 ≈ 6,363 people/km² - Lagos: 15,400,000 ÷ 

replan cycles: 2
steps left   : (none)


### 6. The plan is a reviewable artifact

This is a benefit ReAct structurally cannot offer. Because the plan exists as data
*before* any tool runs, you can gate on it - which connects directly to
`04_human_in_the_loop`. Here is the shape of that gate.

In [11]:
draft = plan_step({"task": TASK, "plan": [], "past_steps": [], "response": "", "cycles": 0})

print()
print("PROPOSED PLAN (what you would show a human before spending money):")
for i, s in enumerate(draft["plan"], 1):
    print("  %d. %s" % (i, s))
print()
print("steps mentioning a tool:",
      sum(1 for s in draft["plan"] if "population" in s.lower() or "area" in s.lower()))
print()
print("In production: interrupt() here, let a reviewer edit the list, then resume.")
print("See 04_human_in_the_loop for the interrupt/resume mechanics.")

[planner  ] 4 steps
            1. Retrieve the population and area in square kilometers for Tokyo, Lagos, and Lima using the `pop
            2. Calculate the population density (population divided by area) for each of the three cities.
            3. Identify the city with the highest calculated population density and round its density to the n
            4. State the answer as the city name and the rounded density.

PROPOSED PLAN (what you would show a human before spending money):
  1. Retrieve the population and area in square kilometers for Tokyo, Lagos, and Lima using the `population(city)` and `area_km2(city)` tools.
  2. Calculate the population density (population divided by area) for each of the three cities.
  3. Identify the city with the highest calculated population density and round its density to the nearest whole number.
  4. State the answer as the city name and the rounded density.

steps mentioning a tool: 3

In production: interrupt() here, let a reviewer edit th

### 7. Failure modes

**Over-planning.** Ask a model to plan and it will happily produce eleven steps for
a two-step problem. Every extra step is an extra executor call. *Fix:* say "minimal"
and "do not add steps that are not needed" (we do), and cap the list length (we do).

**Brittle plans.** A plan written with zero information assumes the world cooperates.
*Fix:* the replanner. It is not optional.

**Replanner loops.** The replanner keeps producing "one more step" and never says
ANSWER. *Fix:* `MAX_CYCLES` plus a `force_answer` node. Never let a graph decide its
own termination without a budget.

**Losing the objective.** The executor sees a step and forgets what it was for.
*Fix:* always include the overall objective in the executor prompt (we do).

**Context growth in `past_steps`.** After 20 steps the history is large again - the
very problem we left ReAct to escape. *Fix:* pass only the last N (we pass 4), or
summarise older steps.

In [12]:
print("Guards in this implementation:")
print("  MAX_CYCLES         =", MAX_CYCLES)
print("  plan length capped =", 8, "(planner),", 6, "(replanner)")
print("  past_steps shown   = last 4 only")
print("  force_answer node  = yes (never exits without an answer)")
print()
print("actual cycles used :", out["cycles"], "of", MAX_CYCLES)
print("actual steps run   :", len(out["past_steps"]))

Guards in this implementation:
  MAX_CYCLES         = 6
  plan length capped = 8 (planner), 6 (replanner)
  past_steps shown   = last 4 only
  force_answer node  = yes (never exits without an answer)

actual cycles used : 2 of 6
actual steps run   : 2


### 8. Plan-and-Execute vs ReAct

| | ReAct | Plan-and-Execute |
|---|---|---|
| Plan visible before execution | No | **Yes** |
| Human approval possible | Hard | **Natural** |
| Cheap model for the grunt work | No | **Yes** |
| Adapts to surprises | Yes, inherently | Only via the replanner |
| Extra LLM calls | 0 | 1 planner + 1 replanner per step |
| Best for | Short, exploratory tasks | Multi-step tasks with a knowable shape |

The honest trade: Plan-and-Execute **adds** LLM calls. It pays for itself when the
executor can be a cheaper model, when the plan can be cached or reviewed, or when
the task is long enough that ReAct's context growth dominates.

### Recap

- The planner writes all steps up front; the executor runs them one at a time; the
  replanner decides whether to stop or revise.
- Consume the step you executed (`plan[1:]`) or you will loop forever.
- The replanner is what makes this adaptive rather than a rigid script.
- Always cap cycles and always have a forced-answer path.

### Next

**[04_rewoo](04_rewoo.ipynb)** - keep the plan, delete the per-step LLM calls.